**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 LangChain 소개 및 기본 실습

## Part 1 | Session 5: LangChain 모듈, LCEL, Memory, 챗봇

---

### 📋 학습 목표

- 🔹 LangChain의 개념과 주요 모듈을 이해합니다
- 🔹 LLM/ChatModel의 기본 사용법을 익힙니다
- 🔹 ChatPromptTemplate으로 프롬프트를 관리합니다
- 🔹 LCEL(LangChain Expression Language)로 파이프라인을 구성합니다
- 🔹 Output Parser로 출력을 구조화합니다
- 🔹 Memory를 활용하여 대화 이력을 관리합니다
- 🔹 간단한 챗봇을 구현합니다

### 📦 필요 라이브러리

```
langchain, langchain-openai, langchain-core, python-dotenv
```

### ⏱️ 예상 소요 시간: 약 90분

---

In [ ]:
# 필요 라이브러리 설치
# !pip install langchain langchain-openai langchain-core python-dotenv

import os
from dotenv import load_dotenv

# .env 파일에서 환경변수 로드
load_dotenv()

# API Key 확인
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("✅ OpenAI API Key가 설정되었습니다!")
    print(f"   Key 앞 8자: {api_key[:8]}...")
else:
    print("❌ API Key가 설정되지 않았습니다!")
    print("   .env 파일에 OPENAI_API_KEY=sk-... 를 추가해주세요.")

# LangChain 버전 확인
import langchain
print(f"\n📦 langchain 버전: {langchain.__version__}")

---

## 🎯 LangChain 소개 및 주요 모듈

### LangChain이란?

LangChain은 LLM 기반 어플리케이션을 쉽게 개발할 수 있도록 도와주는 **프레임워크**입니다.

### LangChain의 핵심 가치

- 🔸 **모듈화**: 각 기능을 독립된 모듈로 제공
- 🔸 **조합 가능**: 모듈을 파이프라인으로 연결
- 🔸 **확장 가능**: 다양한 LLM, 벡터DB, 도구 지원
- 🔸 **추상화**: 복잡한 로직을 간단한 인터페이스로

### 주요 모듈 구조

```
LangChain 생태계
  │
  ├── langchain-core       : 핵심 추상화 (LCEL, 메시지, 프롬프트)
  ├── langchain            : 체인, 에이전트, 메모리
  ├── langchain-openai     : OpenAI 통합
  ├── langchain-community  : 커뮤니티 통합
  ├── langgraph            : 에이전트 워크플로우
  └── langsmith            : 디버깅/모니터링
```

### 핵심 구성요소

| 구성요소 | 설명 | 예시 |
|---------|------|------|
| Model | LLM 모델 래퍼 | ChatOpenAI, ChatOllama |
| Prompt | 프롬프트 템플릿 | ChatPromptTemplate |
| Output Parser | 출력 파싱 | StrOutputParser, JsonOutputParser |
| Chain | 모듈 조합 | LCEL 파이프라인 |
| Memory | 대화 이력 관리 | ConversationBufferMemory |
| Retriever | 문서 검색 | VectorStoreRetriever |
| Agent | 도구 활용 자율 행동 | ReAct Agent |

---

---

## 1️⃣ LLM/ChatModel 기본 사용

LangChain에서 LLM 모델을 사용하는 방법을 알아봅니다.

### ChatModel vs LLM

| 구분 | ChatModel | LLM |
|------|-----------|-----|
| 입력 | 메시지 리스트 | 문자열 |
| 출력 | AIMessage 객체 | 문자열 |
| 예시 | ChatOpenAI | OpenAI (deprecated) |

최신 LangChain에서는 **ChatModel**을 사용하는 것이 권장됩니다.

---

In [ ]:
# ChatOpenAI 기본 사용
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

print("=" * 60)
print("🤖 ChatOpenAI 기본 사용")
print("=" * 60)

# ChatModel 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=300,
)

print("✅ ChatOpenAI 모델 생성 완료!")
print(f"   모델: {llm.model_name}")
print(f"   Temperature: {llm.temperature}")

# 방법 1: invoke() - 가장 기본적인 호출
print("\n--- 방법 1: invoke() ---")
response = llm.invoke("LangChain이 뭔지 한 문장으로 설명해줘.")
print(f"🤖 응답 타입: {type(response).__name__}")
print(f"🤖 응답: {response.content}")

In [ ]:
# 메시지 객체 사용
print("=" * 60)
print("📨 메시지 객체 사용")
print("=" * 60)

# 방법 2: 메시지 리스트로 호출
messages = [
    SystemMessage(content="당신은 파이썬 전문가입니다. 한국어로 간결하게 답변하세요."),
    HumanMessage(content="리스트 컴프리헨션이 뭐야?"),
]

response = llm.invoke(messages)
print(f"\n🤖 응답:\n{response.content}")

# 응답 메타데이터
print(f"\n📊 메타데이터:")
print(f"   토큰 사용: {response.usage_metadata}")
print(f"   응답 ID: {response.id}")

In [ ]:
# 스트리밍 사용
print("=" * 60)
print("🌊 LangChain 스트리밍")
print("=" * 60)

print("\n🤖 스트리밍 응답:")
print("-" * 40)

# stream() 메서드로 스트리밍
for chunk in llm.stream("파인튜닝의 장점을 3가지 알려줘."):
    print(chunk.content, end="", flush=True)

print("\n" + "-" * 40)
print("\n✅ stream() 메서드로 간편하게 스트리밍 가능!")

---

## 2️⃣ 프롬프트 템플릿 (ChatPromptTemplate)

프롬프트를 **재사용 가능한 템플릿**으로 관리합니다.

### 왜 프롬프트 템플릿이 필요한가?

- 🔸 프롬프트의 **재사용성** 향상
- 🔸 **변수 치환**으로 동적 프롬프트 생성
- 🔸 프롬프트 **버전 관리** 용이
- 🔸 **일관된 형식** 유지

---

In [ ]:
# ChatPromptTemplate 기본 사용
from langchain_core.prompts import ChatPromptTemplate

print("=" * 60)
print("📝 ChatPromptTemplate 기본 사용")
print("=" * 60)

# 방법 1: from_messages로 생성
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다. {language}로 답변하세요."),
    ("human", "{question}"),
])

print("📌 템플릿 변수:")
print(f"   {prompt.input_variables}")

# 변수 치환하여 메시지 생성
messages = prompt.format_messages(
    role="머신러닝",
    language="한국어",
    question="오버피팅을 방지하는 방법은?"
)

print(f"\n📨 생성된 메시지:")
for msg in messages:
    print(f"   [{msg.type}] {msg.content}")

# LLM 호출
response = llm.invoke(messages)
print(f"\n🤖 응답:\n{response.content}")

In [ ]:
# 다양한 프롬프트 템플릿 예시
print("=" * 60)
print("📝 실용적인 프롬프트 템플릿 예시")
print("=" * 60)

# 번역 템플릿
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "{source_lang}를 {target_lang}로 번역하세요. 번역 결과만 출력하세요."),
    ("human", "{text}"),
])

# 요약 템플릿
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 텍스트를 {num_sentences}문장으로 요약하세요."),
    ("human", "{text}"),
])

# 번역 실행
print("\n--- 번역 ---")
msg = translate_prompt.format_messages(
    source_lang="한국어",
    target_lang="영어",
    text="파인튜닝은 사전 학습된 모델을 특정 작업에 맞게 추가 학습하는 기법입니다."
)
response = llm.invoke(msg)
print(f"🤖 {response.content}")

# 요약 실행
print("\n--- 요약 ---")
msg = summary_prompt.format_messages(
    num_sentences=2,
    text="""트랜스포머는 2017년 구글에서 발표한 신경망 아키텍처입니다. 
    기존 RNN/LSTM의 순차적 처리 한계를 극복하고, 셀프 어텐션 메커니즘을 통해 
    입력 시퀀스의 모든 위치를 동시에 참조할 수 있습니다. 이를 통해 병렬 처리가 가능해져 
    학습 속도가 크게 향상되었으며, GPT, BERT, T5 등 현대 LLM의 기반이 되었습니다."""
)
response = llm.invoke(msg)
print(f"🤖 {response.content}")

print("\n✅ 템플릿을 미리 정의하면 일관된 프롬프트를 재사용할 수 있습니다!")

---

## 3️⃣ LCEL (LangChain Expression Language) 파이프라인

LCEL은 LangChain의 **파이프라인 구성 문법**입니다. `|` (파이프) 연산자로 모듈을 연결합니다.

### LCEL의 핵심 개념

```python
chain = prompt | model | output_parser
#       입력     →    처리     →    출력 파싱
```

### LCEL의 장점

- 🔸 **직관적 구문**: `|`로 데이터 흐름을 표현
- 🔸 **자동 스트리밍**: chain.stream()으로 자동 지원
- 🔸 **비동기 지원**: chain.ainvoke()로 비동기 실행
- 🔸 **배치 처리**: chain.batch()로 대량 처리
- 🔸 **재시도/폴백**: with_retry(), with_fallbacks()

---

In [ ]:
# LCEL 기본 파이프라인
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

print("=" * 60)
print("🔗 LCEL 기본 파이프라인")
print("=" * 60)

# 파이프라인 구성: 프롬프트 → 모델 → 출력 파서
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {topic} 전문가입니다. 한국어로 간결하게 답변하세요."),
    ("human", "{question}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
output_parser = StrOutputParser()

# LCEL 체인 생성 ( | 연산자로 연결)
chain = prompt | model | output_parser

print("✅ LCEL 체인 생성 완료!")
print(f"   체인 구조: prompt → model → output_parser")

# invoke() 호출
result = chain.invoke({
    "topic": "딥러닝",
    "question": "CNN과 RNN의 차이점을 설명해주세요."
})

print(f"\n📝 입력: topic='딥러닝', question='CNN과 RNN의 차이점을 설명해주세요.'")
print(f"\n🤖 출력 (문자열):")
print(result)
print(f"\n📊 출력 타입: {type(result).__name__}")
print("💡 StrOutputParser가 AIMessage를 문자열로 변환했습니다!")

In [ ]:
# LCEL 스트리밍과 배치 처리
print("=" * 60)
print("🔗 LCEL 스트리밍 & 배치 처리")
print("=" * 60)

# 스트리밍
print("\n--- 스트리밍 ---")
print("🤖 ", end="")
for chunk in chain.stream({"topic": "파이썬", "question": "데코레이터를 간단히 설명해줘."}):
    print(chunk, end="", flush=True)
print()

# 배치 처리
print("\n--- 배치 처리 ---")
questions = [
    {"topic": "파이썬", "question": "GIL이 뭐야?"},
    {"topic": "딥러닝", "question": "드롭아웃이 뭐야?"},
    {"topic": "데이터", "question": "정규화가 뭐야?"},
]

results = chain.batch(questions)

for q, r in zip(questions, results):
    print(f"\n📝 [{q['topic']}] {q['question']}")
    print(f"🤖 {r[:100]}...")

print(f"\n✅ batch()로 {len(questions)}개의 질문을 한 번에 처리했습니다!")

In [ ]:
# 체인 연결: 다단계 파이프라인
print("=" * 60)
print("🔗 다단계 LCEL 파이프라인")
print("=" * 60)

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 1단계: 질문에 대한 답변 생성
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 질문에 한국어로 상세히 답변하세요."),
    ("human", "{question}"),
])

# 2단계: 답변을 요약
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 텍스트를 핵심만 2줄로 요약하세요."),
    ("human", "{text}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
parser = StrOutputParser()

# 1단계 체인
answer_chain = answer_prompt | model | parser

# 2단계 체인 (1단계 결과를 입력으로)
full_chain = (
    answer_chain  # 1단계: 답변 생성
    | RunnableLambda(lambda x: {"text": x})  # 결과를 dict로 변환
    | summary_prompt  # 2단계: 요약 프롬프트
    | model  # 2단계: 모델 호출
    | parser  # 2단계: 문자열 파싱
)

# 실행
question = "트랜스포머의 셀프 어텐션 메커니즘이 뭐야?"
print(f"\n📝 질문: {question}")

# 1단계 결과
print(f"\n--- 1단계: 상세 답변 ---")
answer = answer_chain.invoke({"question": question})
print(f"🤖 {answer}")

# 2단계 결과 (전체 파이프라인)
print(f"\n--- 2단계: 요약 ---")
summary = full_chain.invoke({"question": question})
print(f"🤖 {summary}")

print("\n✅ 다단계 파이프라인으로 '답변 생성 → 요약'을 자동화했습니다!")

---

## 4️⃣ Output Parser 활용

Output Parser는 LLM 출력을 **원하는 형태**로 변환합니다.

### 주요 Output Parser

| Parser | 설명 | 출력 타입 |
|--------|------|----------|
| StrOutputParser | 문자열로 변환 | str |
| JsonOutputParser | JSON으로 파싱 | dict |
| CommaSeparatedListOutputParser | 쉼표 구분 리스트 | list |
| PydanticOutputParser | Pydantic 모델로 파싱 | Pydantic 객체 |

---

In [ ]:
# 다양한 Output Parser 활용
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser

print("=" * 60)
print("📤 Output Parser 활용")
print("=" * 60)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. CommaSeparatedListOutputParser
print("\n--- 1. CommaSeparatedListOutputParser ---")
list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "답변을 쉼표로 구분된 리스트로 작성하세요. {format_instructions}"),
    ("human", "{question}"),
])

list_chain = list_prompt | model | list_parser

result = list_chain.invoke({
    "question": "인기 있는 딥러닝 프레임워크 5가지를 알려줘",
    "format_instructions": list_parser.get_format_instructions(),
})

print(f"📊 결과 타입: {type(result).__name__}")
print(f"📊 결과: {result}")
print(f"📊 첫 번째 항목: {result[0]}")

# 2. JsonOutputParser
print("\n--- 2. JsonOutputParser ---")
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """다음 형식의 JSON으로 응답하세요:
{{"name": "모델명", "organization": "개발사", "parameters": "파라미터수", "year": 연도}}
{format_instructions}"""),
    ("human", "{question}"),
])

json_chain = json_prompt | model | json_parser

result = json_chain.invoke({
    "question": "GPT-4에 대해 알려줘",
    "format_instructions": json_parser.get_format_instructions(),
})

print(f"📊 결과 타입: {type(result).__name__}")
print(f"📊 결과: {result}")
print(f"📊 모델명: {result.get('name')}")

print("\n✅ Output Parser로 LLM 출력을 프로그래밍에 바로 활용할 수 있습니다!")

---

## 5️⃣ Memory와 대화 이력 관리

LLM API는 기본적으로 **상태를 저장하지 않습니다(stateless)**.

LangChain의 Memory 모듈을 사용하면 **대화 이력을 자동으로 관리**할 수 있습니다.

### 주요 Memory 유형

| Memory | 설명 | 사용 사례 |
|--------|------|----------|
| ConversationBufferMemory | 전체 대화 저장 | 짧은 대화 |
| ConversationBufferWindowMemory | 최근 N턴만 저장 | 일반적인 챗봇 |
| ConversationSummaryMemory | 대화를 요약하여 저장 | 긴 대화 |
| ConversationTokenBufferMemory | 토큰 수 기반 관리 | 비용 최적화 |

### 최신 LangChain 방식: 메시지 히스토리 직접 관리

최신 LangChain에서는 `RunnableWithMessageHistory`나 직접 메시지 리스트를 관리하는 방식이 권장됩니다.

---

In [ ]:
# 메시지 히스토리 직접 관리 방식 (최신 LangChain 권장)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

print("=" * 60)
print("💾 메시지 히스토리를 활용한 대화 관리")
print("=" * 60)

# MessagesPlaceholder를 사용한 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요."),
    MessagesPlaceholder(variable_name="history"),  # 대화 이력이 여기에 삽입
    ("human", "{input}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser()
chain = prompt | model | parser

# 대화 이력 저장소
chat_history = []

def chat_with_history(user_input):
    """대화 이력을 관리하면서 채팅"""
    # 체인 실행
    response = chain.invoke({
        "history": chat_history,
        "input": user_input,
    })
    
    # 대화 이력에 추가
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))
    
    return response

# 대화 진행
print("\n👤 1턴: 나는 파이썬을 공부하고 있어.")
print(f"🤖 {chat_with_history('나는 파이썬을 공부하고 있어.')}")

print("\n" + "-" * 40)
print("\n👤 2턴: 어떤 프레임워크를 배우면 좋을까?")
print(f"🤖 {chat_with_history('어떤 프레임워크를 배우면 좋을까?')}")

print("\n" + "-" * 40)
print("\n👤 3턴: 내가 지금 뭘 공부하고 있다고 했지?")
print(f"🤖 {chat_with_history('내가 지금 뭘 공부하고 있다고 했지?')}")

print(f"\n📊 대화 이력: {len(chat_history)}개 메시지")
print("✅ 이전 대화 맥락을 기억하고 있습니다!")

In [ ]:
# 대화 이력 확인 및 관리 전략
print("=" * 60)
print("📋 대화 이력 확인")
print("=" * 60)

for i, msg in enumerate(chat_history):
    role = "👤 User" if isinstance(msg, HumanMessage) else "🤖 AI"
    content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"\n[{i}] {role}: {content}")

print(f"\n{'=' * 60}")
print("💡 대화 이력 관리 전략:")
print("   1. 전체 보관: 짧은 대화에 적합")
print("   2. 윈도우: 최근 N턴만 유지 (메모리/비용 절약)")
print("   3. 요약: 오래된 대화를 요약하여 보관")
print("   4. 토큰 제한: 최대 토큰 수 기준으로 관리")

# 윈도우 방식 예시
MAX_HISTORY = 4  # 최근 4개 메시지만 유지 (2턴)
if len(chat_history) > MAX_HISTORY:
    trimmed = chat_history[-MAX_HISTORY:]
    print(f"\n📌 윈도우 적용 예시: {len(chat_history)}개 → {len(trimmed)}개")

---

## 6️⃣ 간단한 챗봇 구현

지금까지 배운 내용을 종합하여 **대화형 챗봇**을 구현합니다.

### 챗봇 구성요소

1. **시스템 프롬프트**: 챗봇의 역할 정의
2. **프롬프트 템플릿**: MessagesPlaceholder로 이력 관리
3. **LCEL 체인**: prompt | model | parser
4. **대화 이력**: 최근 N턴 윈도우 관리

---

In [ ]:
# 간단한 챗봇 클래스 구현
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI

print("=" * 60)
print("🤖 간단한 챗봇 구현")
print("=" * 60)

class SimpleChatbot:
    """LangChain 기반 간단한 챗봇"""
    
    def __init__(self, system_prompt, model_name="gpt-4o-mini", 
                 temperature=0.7, max_history=10):
        self.history = []
        self.max_history = max_history
        
        # 프롬프트 템플릿
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ])
        
        # 모델
        self.model = ChatOpenAI(
            model=model_name,
            temperature=temperature,
            max_tokens=500,
        )
        
        # 체인 구성
        self.chain = self.prompt | self.model | StrOutputParser()
        
        print(f"✅ 챗봇 초기화 완료!")
        print(f"   모델: {model_name}")
        print(f"   최대 이력: {max_history}개 메시지")
    
    def chat(self, user_input):
        """사용자 입력에 대한 응답 생성"""
        # 체인 실행
        response = self.chain.invoke({
            "history": self.history,
            "input": user_input,
        })
        
        # 이력 추가
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=response))
        
        # 이력 크기 관리 (윈도우)
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
        
        return response
    
    def stream_chat(self, user_input):
        """스트리밍 응답 생성"""
        full_response = ""
        for chunk in self.chain.stream({"history": self.history, "input": user_input}):
            full_response += chunk
            yield chunk
        
        # 이력 추가
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=full_response))
        
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
    
    def reset(self):
        """대화 이력 초기화"""
        self.history = []
        print("🔄 대화 이력이 초기화되었습니다.")
    
    def get_history_count(self):
        """현재 대화 이력 수"""
        return len(self.history)

# 챗봇 생성
bot = SimpleChatbot(
    system_prompt="""당신은 AI/ML 학습을 도와주는 친절한 튜터입니다.
규칙:
- 한국어로 답변합니다.
- 초보자도 이해할 수 있게 쉽게 설명합니다.
- 필요하면 예시 코드를 포함합니다.
- 답변은 간결하게 유지합니다.""",
    max_history=10,
)

In [ ]:
# 챗봇과 대화하기
print("=" * 60)
print("💬 챗봇과 대화하기")
print("=" * 60)

# 대화 1
print("\n👤 User: LoRA가 뭐야?")
print(f"🤖 Bot: {bot.chat('LoRA가 뭐야?')}")

# 대화 2
print(f"\n{'─' * 50}")
print("\n👤 User: 그거랑 QLoRA는 뭐가 달라?")
print(f"🤖 Bot: {bot.chat('그거랑 QLoRA는 뭐가 달라?')}")

# 대화 3 (스트리밍)
print(f"\n{'─' * 50}")
print("\n👤 User: 내 GPU가 8GB인데 어떤 모델까지 학습 가능해?")
print("🤖 Bot: ", end="")
for chunk in bot.stream_chat("내 GPU가 8GB인데 어떤 모델까지 학습 가능해?"):
    print(chunk, end="", flush=True)
print()

print(f"\n📊 현재 대화 이력: {bot.get_history_count()}개 메시지")

In [ ]:
# 대화 맥락 유지 확인
print("=" * 60)
print("🔄 대화 맥락 유지 확인")
print("=" * 60)

print("\n👤 User: 앞에서 설명해준 내용을 요약해줘.")
print(f"🤖 Bot: {bot.chat('앞에서 설명해준 내용을 요약해줘.')}")

print(f"\n📊 대화 이력: {bot.get_history_count()}개 메시지")
print("\n✅ 이전 대화 맥락을 기억하고 요약할 수 있습니다!")

# 대화 초기화
bot.reset()

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 내용 |
|------|----------|
| LangChain 구조 | langchain-core, langchain-openai 등 모듈화된 구조 |
| ChatModel | ChatOpenAI로 모델 생성, invoke/stream/batch 지원 |
| PromptTemplate | ChatPromptTemplate으로 재사용 가능한 프롬프트 관리 |
| LCEL | `\|` 연산자로 prompt → model → parser 파이프라인 구성 |
| Output Parser | StrOutputParser, JsonOutputParser 등으로 출력 구조화 |
| Memory | MessagesPlaceholder + 대화 이력 리스트로 맥락 유지 |
| 챗봇 | 위 요소들을 조합하여 대화형 AI 구현 |

### LCEL 핵심 정리

```python
# 기본 체인
chain = prompt | model | parser

# 실행 방법
chain.invoke({...})   # 단일 실행
chain.stream({...})   # 스트리밍
chain.batch([...])    # 배치 처리
```

### 다음 파트 예고

- 🔜 Part 2: 모델 서빙(Ollama, Transformers) & RAG 파이프라인
- 🔜 로컬 모델 실행과 벡터 DB 활용을 배웁니다!

---

### 💡 실습 과제

1. SimpleChatbot 클래스를 활용하여 자신만의 전문 분야 챗봇을 만들어보세요.
   - 예: 요리 챗봇, 여행 가이드, 코딩 멘토 등
2. LCEL 파이프라인을 확장하여 '질문 → 답변 → 영어 번역' 3단계 체인을 만들어보세요.
3. JsonOutputParser를 활용하여 뉴스 기사에서 '제목, 날짜, 핵심 키워드, 요약'을 추출하는 체인을 만들어보세요.
4. (선택) ConversationSummaryMemory 방식을 구현해보세요: 5턴 이상의 오래된 대화를 요약하여 저장.

---